In [ ]:
import os
import laspy
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [ ]:
INPUT_DIR  = ""
OUTPUT_DIR = ""

PATCH_SIZE             = 20.0
STRIDE                 = 10.0
CLASSIFICATION_ATTRIBUTE = "classification"
CLASSES_TO_EXCLUDE     = [3]
# 0: unclassified
# 1: low_vegetation
# 2: terrain
# 3: out-points
# 4: stem
# 5: live branches
# 6: woody_branches
MIN_TOTAL_POINTS       = 100    # patch odrzucany jeśli ma mniej punktów
MAX_TOTAL_POINTS       = 50000  

In [ ]:
def make_starts(coord_min, coord_max, patch_size, stride):
    starts = list(np.arange(coord_min, coord_max - patch_size, stride))
    last = coord_max - patch_size
    if not starts or starts[-1] < last - 1e-6:
        starts.append(last)
    return starts


def cut_patches(input_dir, output_dir):
    os.makedirs(output_dir, exist_ok=True)
    files = list(Path(input_dir).glob("*.las")) + list(Path(input_dir).glob("*.laz"))
    if not files:
        print(f"ERROR: Brak plików w {input_dir}")
        return

    total_kept, total_disc = 0, 0

    for file_path in files:
        try:
            print(f"\nPlik: {file_path.name}")
            src = laspy.read(file_path)
            xyz    = src.xyz.copy()
            points = src.points.copy()

            x_min, y_min = xyz[:, 0].min(), xyz[:, 1].min()
            x_max, y_max = xyz[:, 0].max(), xyz[:, 1].max()

            kept, disc = 0, 0
            base = file_path.stem

            x_starts = make_starts(x_min, x_max, PATCH_SIZE, STRIDE)
            y_starts = make_starts(y_min, y_max, PATCH_SIZE, STRIDE)

            for x0 in tqdm(x_starts, desc=f"  {base}"):
                for y0 in y_starts:
                    x1, y1 = x0 + PATCH_SIZE, y0 + PATCH_SIZE

                    mask = (
                        (xyz[:, 0] >= x0) & (xyz[:, 0] < x1) &
                        (xyz[:, 1] >= y0) & (xyz[:, 1] < y1)
                    )
                    patch = points[mask]

                    if CLASSES_TO_EXCLUDE:
                        keep = np.isin(patch[CLASSIFICATION_ATTRIBUTE], CLASSES_TO_EXCLUDE, invert=True)
                        patch = patch[keep]

                    if len(patch) < MIN_TOTAL_POINTS:
                        disc += 1
                        continue

                    if MAX_TOTAL_POINTS and len(patch) > MAX_TOTAL_POINTS:
                        idx   = np.random.choice(len(patch), MAX_TOTAL_POINTS, replace=False)
                        patch = patch[idx]

                    kept += 1
                    out = laspy.LasData(src.header)
                    out.points = patch
                    out.write(os.path.join(output_dir, f"{base}_patch_{kept:04d}.laz"))

            total_kept += kept
            total_disc += disc
            print(f"  -> zachowano {kept}, odrzucono {disc}")

        except Exception as e:
            print(f"  ERROR {file_path.name}: {e}")

    print(f"\nGotowe. Łącznie zachowano: {total_kept}, odrzucono: {total_disc}")

In [ ]:
cut_patches(INPUT_DIR, OUTPUT_DIR)